In [40]:
import pandas as pd
import numpy as np
import os

In [41]:
pasta_dados = "..\\data\\input"

dados_campanha = os.path.join(pasta_dados, 'Campanha Incentivo - Distribuição Vinho.xlsx')
dados_configuracao = os.path.join(pasta_dados, 'Configuracao da Campanha.xlsx')
dados_brutos = os.path.join(pasta_dados, 'Dados_Brutos.csv')

In [42]:
relatorio_campanha_premiacao = pd.read_excel(dados_campanha, sheet_name="Premiação Distribuidores")
relatorio_campanha_detalhamento = pd.read_excel(dados_campanha, sheet_name="Detalhamento Mês Apurado")
relatorio_configuracao_cliente = pd.read_excel(dados_configuracao, sheet_name="Cliente")
relatorio_configuracao_produto = pd.read_excel(dados_configuracao, sheet_name="Produto")
relatorio_bruto = pd.read_csv(dados_brutos, sep=";", encoding="utf-8", dtype={"NUMERODOCUMENTO": "str"})

In [43]:
relatorio_bruto.info()

<class 'pandas.DataFrame'>
RangeIndex: 139741 entries, 0 to 139740
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   CODIGOCLIENTE             139741 non-null  int64
 1   DESCRICAOCLIENTE          139741 non-null  str  
 2   CNPJ                      139741 non-null  str  
 3   CODIGOGRUPOCLIENTE        139741 non-null  int64
 4   DESCRICAOGRUPOCLIENTE     139741 non-null  str  
 5   CODIGOFAMILIA             139741 non-null  int64
 6   DESCRICAOFAMILIA          139741 non-null  str  
 7   CODIGOGRUPOPRODUTO        139741 non-null  int64
 8   DESCRICAOGRUPOPRODUTO     139741 non-null  str  
 9   CODIGOPRODUTO             139741 non-null  int64
 10  NOMEPRODUTO               139741 non-null  str  
 11  CODIGODOCUMENTO           139741 non-null  int64
 12  NUMERODOCUMENTO           139741 non-null  str  
 13  SERIEDOCUMENTO            139741 non-null  int64
 14  DATAEMISSAO               13974

### 1. Filtragem:

In [ ]:
CFOPS_CAMPANHA = (5102, 5106, 5110, 6102, 6106, 6110, 5160, 6160, 6910, 5910)

In [ ]:
relatorio_apurado = relatorio_bruto[
    (
        relatorio_bruto["CODIGOCLIENTE"].isin(
            relatorio_configuracao_cliente["CODIGOCLIENTE"]
        )
    )
    &
    (
        relatorio_bruto["CODIGOPRODUTO"].isin(
            relatorio_configuracao_produto["CODIGOPRODUTO"]
        )
    )
    &
    (
        relatorio_bruto["CFOP"].isin(CFOPS_CAMPANHA)
    )
]

relatorio_apurado.shape

(566, 22)

### Validação: 
    - Base Bruta x Base Apurada

In [49]:
relatorio_apurado = relatorio_apurado.rename(columns={
    "VOLUME": "VOLUMEBRUTO"
})

relatorio_apurado.columns

Index(['CODIGOCLIENTE', 'DESCRICAOCLIENTE', 'CNPJ', 'CODIGOGRUPOCLIENTE',
       'DESCRICAOGRUPOCLIENTE', 'CODIGOFAMILIA', 'DESCRICAOFAMILIA',
       'CODIGOGRUPOPRODUTO', 'DESCRICAOGRUPOPRODUTO', 'CODIGOPRODUTO',
       'NOMEPRODUTO', 'CODIGODOCUMENTO', 'NUMERODOCUMENTO', 'SERIEDOCUMENTO',
       'DATAEMISSAO', 'CFOP', 'VALORLIQUIDO', 'VALORBRUTOIMPOSTO',
       'VALORVENDALIQUIDO', 'VOLUMEBRUTO', 'TIPOFATURAMENTO',
       'DESCRICAOTIPOFATURAMENTO'],
      dtype='str')

In [50]:
validacao_campanha_dados_apurados = pd.merge(
    relatorio_campanha_detalhamento,
    relatorio_apurado[["CODIGOCLIENTE", "CODIGODOCUMENTO", "CODIGOPRODUTO", "VOLUMEBRUTO"]],
    how="left",
    on=["CODIGODOCUMENTO", "CODIGOCLIENTE", "CODIGOPRODUTO"]
)

In [52]:
validacao_campanha_dados_apurados

,CODIGOCAMPANHA,DESCRICAOCAMPANHA,CODIGOCLIENTE,NOMECLIENTE,CODIGOGRUPOCLIENTE,CODIGODOCUMENTO,NUMERODOCUMENTO,NUMEROSERIE,DATAEMISSAO,CODIGOFAMILIAPRODUTO,...,CODIGOGRUPOPRODUTO,NOMEGRUPOPRODUTO,CODIGOPRODUTO,NOMEPRODUTO,CODIGOCFOP,DESCRICAOTIPOFATURAMENTO,FATORPRODUTO,VOLUME,DATAAPURACAO,VOLUMEBRUTO
0,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5729304,17369,1,2026-02-20,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,5102,VENDA,1,50,2026-03-09 13:36:59.000,50.0
1,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5738300,17451,1,2026-02-22,80000007,...,90013032,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,5102,VENDA,1,25,2026-03-09 13:36:59.000,25.0
2,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5738300,17451,1,2026-02-22,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,5102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
3,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5684798,17096,1,2026-02-07,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,5102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
4,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5771351,17686,1,2026-02-07,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,6102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5745184,36458,1,2026-02-23,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,6102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
565,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5694289,35822,1,2026-02-10,80000007,...,90013032,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,6102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
566,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5721639,36171,1,2026-02-17,80000007,...,90013032,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,6102,VENDA,1,50,2026-03-09 13:36:58.995,50.0
567,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5745157,36401,1,2026-02-23,80000007,...,90013032,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,6102,VENDA,1,25,2026-03-09 13:36:58.995,25.0


In [53]:
validacao_campanha_dados_apurados["VALIDAÇÃO"] = np.where(
    validacao_campanha_dados_apurados["VOLUME"] != validacao_campanha_dados_apurados["VOLUMEBRUTO"],
    "DIVERGENTE",
    "CORRETO"
)

validacao_campanha_dados_apurados[validacao_campanha_dados_apurados["VALIDAÇÃO"] == "DIVERGENTE"]

,CODIGOCAMPANHA,DESCRICAOCAMPANHA,CODIGOCLIENTE,NOMECLIENTE,CODIGOGRUPOCLIENTE,CODIGODOCUMENTO,NUMERODOCUMENTO,NUMEROSERIE,DATAEMISSAO,CODIGOFAMILIAPRODUTO,...,NOMEGRUPOPRODUTO,CODIGOPRODUTO,NOMEPRODUTO,CODIGOCFOP,DESCRICAOTIPOFATURAMENTO,FATORPRODUTO,VOLUME,DATAAPURACAO,VOLUMEBRUTO,VALIDAÇÃO
14,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5729295,17348,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,DIVERGENTE
15,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5729295,17348,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,DIVERGENTE
114,4002,Harmonização Master - Vinho Tintos Importados ...,10004204,LOGÍSTICA VALE DAS BEBIDAS,2,5728490,136345,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,5110,VENDA,1,25,2026-03-09 13:36:58.995,200.0,DIVERGENTE
183,4002,Harmonização Master - Vinho Tintos Importados ...,10004219,SUPRIMENTOS MASTER BEBIDAS,2,5762186,176536,1,2026-02-19,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,1202,DEVOLUÇÃO DE VENDA,1,-10,2026-03-09 13:36:58.995,NaN,DIVERGENTE
346,4002,Harmonização Master - Vinho Tintos Importados ...,10004227,LOGÍSTICA INTEGRAL DE BEBIDAS,2,5744799,44502,1,2026-02-24,80000007,...,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,2202,DEVOLUÇÃO DE VENDA,1,-50,2026-03-09 13:36:58.995,NaN,DIVERGENTE
559,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5703114,67183,1,2026-02-13,80000007,...,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,DIVERGENTE
560,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5757643,36677,1,2026-02-28,80000007,...,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,2202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,DIVERGENTE
